In [25]:
import pandas as pd
import numpy as np
import sqlite3
import os
import folium
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

- 재난 종류별 특보가 많이 발생하는 지역
- 해당 지역에 적합한 재난 대피소량 -> pass
- 재난 발생이 많은 시간대
- 특보 등급결 재난 발생 비율
- 지역별 대피소 유형
- 지역별 재난 발생 횟수
- 지역별 대피소 총 수용인원

- 대피소 지도 표시
- 재난 발생 지역 표시

In [26]:
df = pd.read_csv("./data/danger_clean.csv", encoding="utf-8-sig")
df.head()

,발표시간,지역,시군구,재난종류,특보등급,해당지역
0,2023-01-01 10:00,경북,군위,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...
1,2023-01-01 10:00,경북,안동,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...
2,2023-01-01 10:00,경북,영주,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...
3,2023-01-01 10:00,경북,의성,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...
4,2023-01-01 10:00,경북,청송,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3545 entries, 0 to 3544
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   발표시간    3545 non-null   str  
 1   지역      3545 non-null   str  
 2   시군구     3545 non-null   str  
 3   재난종류    3545 non-null   str  
 4   특보등급    3545 non-null   str  
 5   해당지역    3545 non-null   str  
dtypes: str(6)
memory usage: 764.4 KB


In [28]:
df["발표시간"] = pd.to_datetime(df["발표시간"])

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3545 entries, 0 to 3544
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   발표시간    3545 non-null   datetime64[us]
 1   지역      3545 non-null   str           
 2   시군구     3545 non-null   str           
 3   재난종류    3545 non-null   str           
 4   특보등급    3545 non-null   str           
 5   해당지역    3545 non-null   str           
dtypes: datetime64[us](1), str(5)
memory usage: 709.8 KB


In [30]:
df["날짜"] = df["발표시간"].dt.date
df["시간"] = df["발표시간"].dt.time
df.head()

,발표시간,지역,시군구,재난종류,특보등급,해당지역,날짜,시간
0,2023-01-01 10:00:00,경북,군위,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...,2023-01-01,10:00:00
1,2023-01-01 10:00:00,경북,안동,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...,2023-01-01,10:00:00
2,2023-01-01 10:00:00,경북,영주,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...,2023-01-01,10:00:00
3,2023-01-01 10:00:00,경북,의성,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...,2023-01-01,10:00:00
4,2023-01-01 10:00:00,경북,청송,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...,2023-01-01,10:00:00


## 분석 내용

### 재난 종류별 특보가 많이 발생한 지역 건수

In [31]:
# 재난 종류별 특보가 많이 발생한 지역 건수
df_type = df[["재난종류","특보등급","시군구"]].dropna()

calamity_type = df_type.groupby(["재난종류", "시군구"]).size().reset_index(name="발생건수").sort_values(["재난종류", "발생건수"], ascending=[True, False])

calamity_type.groupby("재난종류").head(3)

,재난종류,시군구,발생건수
25,강풍,울산,80
14,강풍,부산,66
21,강풍,영덕,66
53,건조,부산,22
64,건조,울산,22
50,건조,대구,21
89,대설,문경,21
91,대설,산청,18
111,대설,함양,18
113,태풍,경산,1


In [32]:
top3

,재난종류,시군구,발생건수
25,강풍,울산,80
14,강풍,부산,66
21,강풍,영덕,66
34,강풍,포항,63
3,강풍,경주,61
53,건조,부산,22
64,건조,울산,22
50,건조,대구,21
74,건조,포항,21
69,건조,창원,20


#### 재난종류별 특보 발생 상위 지역

In [33]:
import plotly.express as px

top3 = calamity_type.groupby("재난종류").head()

fig = px.bar(
    top3,
    x="시군구",
    y="발생건수",
    color="재난종류",
    facet_col="재난종류",
    title="재난종류별 특보 발생 상위 지역",
    color_discrete_map={
        "호우": "#1f77b4",     # 파랑
        "폭염": "#ff7f0e",     # 주황
        "한파": "#17becf",     # 청록
        "강풍": "#2ca02c",     # 초록
        "태풍": "#d62728",     # 빨강
        "대설": "#9467bd",     # 보라
        "건조": "#8c564b",     # 갈색
        "풍랑해일": "#e377c2"  # 핑크
    }
)

fig.show()

#### 재난종류별 특보 발생 지역

In [34]:
fig = px.scatter(
    top3,
    x="재난종류",
    y="시군구",
    size="발생건수",
    color="시군구",
    title="재난종류별 특보 발생 지역",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.show()

### 재난종류별 많이 발생한 날짜 건수

In [35]:
# 재난종류별 많이 발생한 날짜 건수
time_type = (
    df.groupby(["재난종류", "날짜"])
      .size()
      .reset_index(name="발생건수")
      .sort_values(["재난종류", "발생건수"], ascending=[True, False])
)

time_type.groupby("재난종류").head()

,재난종류,날짜,발생건수
108,강풍,2025-03-25,24
99,강풍,2025-02-07,19
26,강풍,2023-07-26,18
65,강풍,2024-03-29,17
33,강풍,2023-11-18,16
231,건조,2026-02-06,27
168,건조,2023-04-12,26
151,건조,2023-02-24,18
234,건조,2026-02-19,17
222,건조,2026-01-01,15


#### 재난종류별 특보 발생 날짜 추이

In [36]:
import plotly.express as px

top_time = time_type.groupby("재난종류").head()

fig = px.line(
    top_time,
    x="날짜",
    y="발생건수",
    color="재난종류",
    markers=True,
    title="재난종류별 특보 발생 날짜 추이",
    
    color_discrete_map={
        "호우": "#1f77b4",   # 파랑
        "폭염": "#ff7f0e",   # 주황
        "한파": "#17becf",   # 청록
        "강풍": "#2ca02c",   # 초록
        "태풍": "#d62728",   # 빨강
        "대설": "#9467bd",   # 보라
        "건조": "#8c564b",   # 갈색
        "풍랑해일": "#e377c2" # 핑크
    }
)

fig.update_layout(
    xaxis_title="날짜",
    yaxis_title="발생건수"
)

fig.show()

#### 재난종류별 특보 발생 날짜

In [37]:
import plotly.express as px

heat = top_time.pivot(
    index="재난종류",
    columns="날짜",
    values="발생건수"
).fillna(0)

# 0 제외 텍스트 생성
text_values = heat.where(heat != 0, "")

fig = px.imshow(
    heat,
    text_auto=False,
    color_continuous_scale="YlOrRd",
    aspect="auto",
    title="재난종류별 특보 발생 날짜"
)

fig.update_traces(
    text=text_values,
    texttemplate="%{text}"
)

fig.show()

### 재난종류별 월별 발생 건수

In [38]:
# 재난종류별 많이 발생한 년월 건수
df["년월"] = df["발표시간"].dt.to_period("M")

result = (
    df.groupby(["재난종류", "년월"])
      .size()
      .reset_index(name="발생건수")
      .sort_values(["재난종류", "발생건수"], ascending=[True, False])
)

result.groupby("재난종류").head()

,재난종류,년월,발생건수
25,강풍,2025-03,59
13,강풍,2024-03,55
6,강풍,2023-07,52
9,강풍,2023-11,37
24,강풍,2025-02,35
52,건조,2026-02,66
51,건조,2026-01,58
46,건조,2025-02,53
35,건조,2023-01,51
36,건조,2023-02,49


In [39]:
df["년월"] = pd.to_datetime(df["발표시간"]).dt.to_period("M").astype(str)

result = (
    df.groupby(["재난종류", "년월"])
      .size()
      .reset_index(name="발생건수")
)

top_month = result.groupby("재난종류").head(5)

#### 재난종류별 월별 발생 추이

In [40]:
import plotly.express as px

fig = px.line(
    top_month,
    x="년월",
    y="발생건수",
    color="재난종류",
    markers=True,
    title="재난종류별 월별 발생 추이",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.update_layout(
    xaxis_title="년월",
    yaxis_title="발생건수"
)

fig.show()

#### 월별 재난 발생 분포

In [41]:
fig = px.bar(
    month_type,
    x="년월",
    y="발생건수",
    color="재난종류",
    title="월별 재난 발생 분포",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.update_layout(barmode="stack")

fig.show()

#### 재난종류별 월별 특보 발생 패턴

In [43]:
import pandas as pd
import plotly.express as px

df["발표시간"] = pd.to_datetime(df["발표시간"])
df["년월"] = df["발표시간"].dt.to_period("M").astype(str)

month_type = (
    df.groupby(["재난종류", "년월"])
      .size()
      .reset_index(name="발생건수")
)

heat = month_type.pivot(
    index="재난종류",
    columns="년월",
    values="발생건수"
).fillna(0)

# 10 이상만 숫자 표시
text_values = heat.applymap(lambda x: f"{int(x)}" if x >= 10 else "")

fig = px.imshow(
    heat,
    color_continuous_scale="YlOrRd",
    aspect="auto",
    title="재난종류별 월별 특보 발생 패턴"
)

fig.update_traces(
    text=text_values,
    texttemplate="%{text}",
    textfont={"size":11, "color":"black"},
    hovertemplate="재난종류: %{y}<br>년월: %{x}<br>발생건수: %{z}<extra></extra>"
)

fig.update_layout(
    width=1400,
    height=500,
    xaxis_title="년월",
    yaxis_title="재난종류",
    coloraxis_colorbar_title="발생건수",
    title_x=0.5
)

fig.show()

AttributeError: 'DataFrame' object has no attribute 'applymap'

### 재난 종류별 특보 발생건수 비율

In [44]:
# 재난종류 + 특보등급별 발생 건수
grade_type = (
    df.groupby(["재난종류", "특보등급"])
      .size()
      .reset_index(name="발생건수")
)

# 비율 계산
grade_type["비율(%)"] = (
    grade_type["발생건수"] / grade_type.groupby("재난종류")["발생건수"].transform("sum") * 100
).round(2)

grade_type

,재난종류,특보등급,발생건수,비율(%)
0,강풍,경보,1,0.14
1,강풍,주의보,723,99.86
2,건조,경보,39,7.43
3,건조,주의보,486,92.57
4,대설,주의보,238,100.00
5,태풍,주의보,20,100.00
6,폭염,경보,93,16.34
7,폭염,주의보,476,83.66
8,폭풍해일,주의보,49,100.00
9,한파,경보,12,3.87


#### 경보 재난 분포율

In [45]:
import plotly.express as px

warning = grade_type[grade_type["특보등급"] == "경보"]

fig_warning = px.pie(
    warning,
    names="재난종류",
    values="발생건수",
    hole=0,
    title="경보 재난 분포",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig_warning.show()

#### 주의보 재난 분포율

In [46]:
advisory = grade_type[grade_type["특보등급"] == "주의보"]

fig_advisory = px.pie(
    advisory,
    names="재난종류",
    values="발생건수",
    hole=0,
    title="주의보 재난 분포",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig_advisory.show()

#### 특보등급별 재난 분포

In [47]:
import plotly.graph_objects as go

# 특보등급별 데이터 분리
warning = grade_type[grade_type["특보등급"] == "경보"].copy()
advisory = grade_type[grade_type["특보등급"] == "주의보"].copy()

# 재난종류별 동일 색상
disaster_colors = {
    "호우": "#1f77b4",
    "강풍": "#d62728",
    "건조": "#ff7f0e",
    "폭염": "#9467bd",
    "한파": "#17becf",
    "대설": "#2ca02c",
    "태풍": "#8c564b",
    "폭풍해일": "#e377c2"
}

# 3% 미만 숨김용 텍스트 함수
def percent_text(values):
    total = sum(values)
    out = []
    for v in values:
        p = v / total * 100
        out.append(f"{p:.1f}%" if p >= 3 else "")
    return out

fig = go.Figure()

# 바깥 도넛: 주의보
fig.add_trace(go.Pie(
    labels=advisory["재난종류"],
    values=advisory["발생건수"],
    hole=0.58,
    sort=False,
    direction="clockwise",
    name="주의보",
    domain=dict(x=[0.12, 0.88], y=[0.12, 0.88]),
    marker=dict(
        colors=[disaster_colors[d] for d in advisory["재난종류"]],
        line=dict(color="white", width=2)
    ),
    text=percent_text(advisory["발생건수"]),
    textinfo="text",
    textposition="inside",
    textfont=dict(size=14, color="black"),
    hovertemplate="주의보<br>%{label}: %{value}건 (%{percent})<extra></extra>",
    showlegend=True,
    legendgroup="disaster"
))

# 안쪽 도넛: 경보
fig.add_trace(go.Pie(
    labels=warning["재난종류"],
    values=warning["발생건수"],
    hole=0.35,
    sort=False,
    direction="clockwise",
    name="경보",
    domain=dict(x=[0.30, 0.70], y=[0.30, 0.70]),
    marker=dict(
        colors=[disaster_colors[d] for d in warning["재난종류"]],
        line=dict(color="white", width=2)
    ),
    text=percent_text(warning["발생건수"]),
    textinfo="text",
    textposition="inside",
    textfont=dict(size=13, color="white"),
    hovertemplate="경보<br>%{label}: %{value}건 (%{percent})<extra></extra>",
    showlegend=False,
    legendgroup="disaster"
))

fig.update_layout(
    title=dict(
        text="특보등급별 재난 분포",
        x=0.5
    ),
    width=900,
    height=650,
    margin=dict(t=80, b=40, l=40, r=180),
    legend=dict(
        title="재난종류",
        x=1.02,
        y=0.5,
        xanchor="left",
        yanchor="middle"
    ),
    annotations=[
    dict(
        x=1.12,
        y=0.8,
        text="안쪽: 경보<br>바깥: 주의보",
        showarrow=False,
        font=dict(size=13),
        align="left"
    )
]
)

fig.show()

- 시도 기준 재난 종류별 특보 발생건수

In [48]:
# 지역 + 특보등급 + 재난종류 발생 건수
result = (
    df.groupby(["지역", "특보등급", "재난종류"])
      .size()
      .reset_index(name="발생건수")
      .sort_values(["지역", "특보등급", "발생건수"], ascending=[True, True, False])
)

result

,지역,특보등급,재난종류,발생건수
1,경남,경보,폭염,46
2,경남,경보,호우,22
0,경남,경보,건조,9
9,경남,주의보,호우,515
3,경남,주의보,강풍,270
4,경남,주의보,건조,227
6,경남,주의보,폭염,189
8,경남,주의보,한파,94
5,경남,주의보,대설,77
7,경남,주의보,폭풍해일,37


### 시도별 재난종류 분포

In [49]:
area_data = (
    result.groupby(["지역","재난종류"])["발생건수"]
    .sum()
    .reset_index()
)

fig = px.area(
    area_data,
    x="지역",
    y="발생건수",
    color="재난종류",
    title="시도별 재난종류 분포",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.show()

#### 지역별 재난 특성 비교

In [50]:
import plotly.graph_objects as go

radar = (
    result.groupby(["지역", "재난종류"])["발생건수"]
    .sum()
    .unstack()
    .fillna(0)
)

colors = {
    "경남": "#1f77b4",
    "경북": "#ff7f0e",
    "대구": "#2ca02c",
    "부산": "#d62728",
    "울산": "#9467bd"
}

fig = go.Figure()

for region in radar.index:
    fig.add_trace(go.Scatterpolar(
        r=radar.loc[region],
        theta=radar.columns,
        fill="toself",
        name=region,
        line=dict(color=colors.get(region, "#333333"))
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    title="지역별 재난 특성 비교"
)

fig.show()

#### 시도 기준 재난 종류별 특보 발생 비율

In [51]:
# 지역 + 재난종류 + 특보등급 발생 건수
result = (
    df.groupby(["지역", "재난종류", "특보등급"])
      .size()
      .reset_index(name="발생건수")
)

# 지역별 비율 계산
result["비율(%)"] = (
    result["발생건수"] /
    result.groupby("지역")["발생건수"].transform("sum") * 100
).round(2)

# 정렬
result = result.sort_values(["지역", "재난종류", "비율(%)"], ascending=[True, True, False])

result

,지역,재난종류,특보등급,발생건수,비율(%)
0,경남,강풍,주의보,270,18.17
2,경남,건조,주의보,227,15.28
1,경남,건조,경보,9,0.61
3,경남,대설,주의보,77,5.18
5,경남,폭염,주의보,189,12.72
4,경남,폭염,경보,46,3.10
6,경남,폭풍해일,주의보,37,2.49
7,경남,한파,주의보,94,6.33
9,경남,호우,주의보,515,34.66
8,경남,호우,경보,22,1.48


In [52]:
fig = px.bar(
    result,
    x="지역",
    y="발생건수",
    color="재난종류",
    facet_col="특보등급",
    title="시도 기준 재난 종류별 특보 발생 건수",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.update_layout(
    barmode="stack",
    yaxis_title="발생건수"
)

fig.show()

### 시도별 재난 종류 비율

In [53]:
result["비율(%)"] = (
    result["발생건수"] /
    result.groupby("지역")["발생건수"].transform("sum") * 100
).round(2)

In [54]:

import plotly.express as px

fig = px.pie(
    result,
    names="재난종류",
    values="비율(%)",
    facet_col="지역",
    color="재난종류",
    title="시도별 재난 종류 비율",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.show()

In [55]:
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['font.family'] = 'Malgun Gothic'                                                                                                                 
plt.rcParams['axes.unicode_minus'] = False 

#### 시도별 재난 특보등급 비율

In [56]:
import plotly.express as px
import pandas as pd

# 지역별 재난 특보등급 비율
fig = px.bar(
    result,
    x="재난종류",
    y="비율(%)",
    color="특보등급",
    facet_col="지역",
    title="시도별 재난 특보등급 비율",

    color_discrete_map={
    "주의보": "#0BF11F",   # 녹색
    "경보": "#FF0000"      # 빨강
}
)

fig.show()

#### 시도별 재난종류 비율

In [57]:
# 시도별 재난종류 비율
pivot = result.pivot_table(
    index="지역",
    columns="재난종류",
    values="비율(%)",
    aggfunc="sum"
)

fig = px.imshow(
    pivot,
    text_auto=True,
    color_continuous_scale="Reds",
    title="시도별 재난종류 비율"
)

fig.show()

In [58]:
import pandas as pd

df_sh = pd.read_csv(".//data/final_shelter_dataset.csv", encoding="utf-8-sig")
df_sh.head()

,대피소명,주소,대피소유형,위도,경도,시도,시군구,지역,수용인원
0,(40417)참바른병원버스정류장,울산광역시 남구 삼산로 246,한파쉼터,35.538289,129.334128,울산,울산,울산 남구,NaN
1,(40404)공업탑버스정류장,울산광역시 남구 삼산로 17,한파쉼터,35.533117,129.309767,울산,울산,울산 남구,NaN
2,(40403)공업탑버스정류장,울산광역시 남구 삼산로 24 (프라자빌딩),한파쉼터,35.532700,129.310409,울산,울산,울산 남구,NaN
3,(40402)공업탑버스정류장,울산광역시 남구 삼산로 11,한파쉼터,35.532990,129.309268,울산,울산,울산 남구,NaN
4,(40401)공업탑버스정류장,울산광역시 남구 삼산로 18,한파쉼터,35.532561,129.309976,울산,울산,울산 남구,NaN


In [59]:
type_region = (
        df_sh.groupby(["시도", "대피소유형"])
              .size()
              .reset_index(name="대피소수")
              .sort_values(["시도", "대피소수"], ascending=[True, False])
)

type_region

,시도,대피소유형,대피소수
3,경남,"한파쉼터,무더위쉼터",6761
0,경남,무더위쉼터,997
1,경남,지진옥외대피장소,710
2,경남,한파쉼터,191
9,경북,"한파쉼터,무더위쉼터",5071
5,경북,지진옥외대피장소,1307
7,경북,지진해일대피장소,344
4,경북,무더위쉼터,314
8,경북,한파쉼터,139
6,경북,"지진옥외대피장소,지진해일대피장소",21


In [60]:
pivot = pd.pivot_table(
    df_sh,
    index="시도",
    columns="대피소유형",
    aggfunc="size",
    fill_value=0
)

pivot

대피소유형,무더위쉼터,지진옥외대피장소,"지진옥외대피장소,지진해일대피장소",지진해일대피장소,한파쉼터,"한파쉼터,무더위쉼터"
시도,,,,,,
경남,997,710,0,0,191,6761
경북,314,1307,21,344,139,5071
대구,455,725,0,0,80,823
부산,513,629,0,67,47,1112
울산,66,256,2,53,157,1031


### 시도별 대피소 유형 분포

In [61]:
import plotly.express as px

fig = px.bar(
    type_region,
    x="시도",
    y="대피소수",
    color="대피소유형",
    title="시도별 대피소 유형 분포",
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

In [62]:
pivot["무더위쉼터_합계"] = pivot["무더위쉼터"] + pivot["한파쉼터,무더위쉼터"]

pivot["지진대피소_합계"] = (
    pivot["지진옥외대피장소"] +
    pivot["지진옥외대피장소,지진해일대피장소"] +
    pivot["지진해일대피장소"]
)

pivot_new = pivot[[
    "무더위쉼터_합계",
    "지진대피소_합계",
    "한파쉼터"
]]

pivot_new

대피소유형,무더위쉼터_합계,지진대피소_합계,한파쉼터
시도,,,
경남,7758,710,191
경북,5385,1672,139
대구,1278,725,80
부산,1625,696,47
울산,1097,311,157


In [63]:
pivot["대피소_합계"] = pivot["무더위쉼터_합계"]+pivot["지진대피소_합계"]+pivot["한파쉼터"]
pi = pivot[[
    "무더위쉼터_합계",
    "지진대피소_합계",
    "한파쉼터",
    "대피소_합계"
]]

pi

대피소유형,무더위쉼터_합계,지진대피소_합계,한파쉼터,대피소_합계
시도,,,,
경남,7758,710,191,8659
경북,5385,1672,139,7196
대구,1278,725,80,2083
부산,1625,696,47,2368
울산,1097,311,157,1565


### 지역별 재난 대피소 횟수

In [64]:
# 지역별 재난 발생 횟수
region_disaster_count = (
    df_sh.groupby("지역")
      .size()
      .reset_index(name="재난발생횟수")
      .sort_values("재난발생횟수", ascending=False)
)

region_disaster_count

,지역,재난발생횟수
39,경북 포항시,1148
12,경남 창원시,1126
10,경남 진주시,843
27,경북 안동시,695
35,경북 의성군,602
...,...,...
63,부산 영도구,66
33,경북 울릉군,47
40,경상북도,19
66,부산광역시,12


In [65]:
pi

대피소유형,무더위쉼터_합계,지진대피소_합계,한파쉼터,대피소_합계
시도,,,,
경남,7758,710,191,8659
경북,5385,1672,139,7196
대구,1278,725,80,2083
부산,1625,696,47,2368
울산,1097,311,157,1565


In [66]:
pi.index.name = "지역"
pi

대피소유형,무더위쉼터_합계,지진대피소_합계,한파쉼터,대피소_합계
지역,,,,
경남,7758,710,191,8659
경북,5385,1672,139,7196
대구,1278,725,80,2083
부산,1625,696,47,2368
울산,1097,311,157,1565


#### 지역별 대피소 유형 및 총 대피소 수

In [75]:
pi.columns

Index(['무더위쉼터_합계', '지진대피소_합계', '한파쉼터', '대피소_합계'], dtype='str', name='대피소유형')

In [72]:
pi.head()

대피소유형,무더위쉼터_합계,지진대피소_합계,한파쉼터,대피소_합계
지역,,,,
경남,7758,710,191,8659
경북,5385,1672,139,7196
대구,1278,725,80,2083
부산,1625,696,47,2368
울산,1097,311,157,1565


In [76]:
pi.index

Index(['경남', '경북', '대구', '부산', '울산'], dtype='str', name='지역')

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
# 무더위쉼터
fig.add_bar(
    x=pi.index,
    y=pi["무더위쉼터_합계"],
    name="무더위쉼터",
    marker_color="#ff7f0e",
    customdata=pi["대피소_합계"],
    hovertemplate=(
        "지역: %{x}<br>"
        "무더위쉼터: %{y}<br>"
        "총 대피소 수: %{customdata}<extra></extra>"
    )
)

# 지진대피소
fig.add_bar(
    x=pi.index,
    y=pi["지진대피소_합계"],
    name="지진대피소",
    marker_color="#1f77b4",
    customdata=pi["대피소_합계"],
    hovertemplate=(
        "지역: %{x}<br>"
        "지진대피소: %{y}<br>"
        "총 대피소 수: %{customdata}<extra></extra>"
    )
)

# 한파쉼터
fig.add_bar(
    x=pi.index,
    y=pi["한파쉼터"],
    name="한파쉼터",
    marker_color="#17becf",
    customdata=pi["대피소_합계"],
    hovertemplate=(
        "지역: %{x}<br>"
        "한파쉼터: %{y}<br>"
        "총 대피소 수: %{customdata}<extra></extra>"
    )
)

fig.update_layout(
    barmode="stack",
    title="지역별 대피소 유형 및 총 대피소 수",
    xaxis_title="지역",
    yaxis_title="대피소 수",
    legend_title="대피소 유형",
    template="plotly_white",
    hovermode="x unified"
)

fig.add_trace(
    go.Scatter(
        x=pi.index,
        y=pi["대피소_합계"],
        mode="text",
        text=pi["대피소_합계"],
        textposition="top center",
        showlegend=False
    )
)
fig.show()

In [80]:
import plotly.graph_objects as go

# ✅ 수정: 선택할 대피소유형
selected_types = ["무더위쉼터_합계", "지진대피소_합계", "한파쉼터"]

fig = go.Figure()

# 무더위쉼터
if "무더위쉼터_합계" in selected_types:  # ✅ 수정
    fig.add_bar(
        x=pi.index,
        y=pi["무더위쉼터_합계"],
        name="무더위쉼터",
        marker_color="#ff7f0e",
        customdata=pi[selected_types].sum(axis=1),  # ✅ 수정
        hovertemplate=(
            "지역: %{x}<br>"
            "무더위쉼터: %{y}<br>"
            "총 대피소 수: %{customdata}<extra></extra>"
        )
    )

# 지진대피소
if "지진대피소_합계" in selected_types:  # ✅ 수정
    fig.add_bar(
        x=pi.index,
        y=pi["지진대피소_합계"],
        name="지진대피소",
        marker_color="#1f77b4",
        customdata=pi[selected_types].sum(axis=1),  # ✅ 수정
        hovertemplate=(
            "지역: %{x}<br>"
            "지진대피소: %{y}<br>"
            "총 대피소 수: %{customdata}<extra></extra>"
        )
    )

# 한파쉼터
if "한파쉼터" in selected_types:  # ✅ 수정
    fig.add_bar(
        x=pi.index,
        y=pi["한파쉼터"],
        name="한파쉼터",
        marker_color="#17becf",
        customdata=pi[selected_types].sum(axis=1),  # ✅ 수정
        hovertemplate=(
            "지역: %{x}<br>"
            "한파쉼터: %{y}<br>"
            "총 대피소 수: %{customdata}<extra></extra>"
        )
    )

fig.update_layout(
    barmode="stack",
    title="지역별 대피소 유형 및 총 대피소 수",
    xaxis_title="지역",
    yaxis_title="대피소 수",
    legend_title="대피소 유형",
    template="plotly_white",
    hovermode="x unified"
)

fig.add_trace(
    go.Scatter(
        x=pi.index,
        y=pi[selected_types].sum(axis=1),  # ✅ 수정
        mode="text",
        text=pi[selected_types].sum(axis=1),  # ✅ 수정
        textposition="top center",
        showlegend=False
    )
)

fig.show()

In [68]:
import plotly.graph_objects as go
import numpy as np

fig = go.FigureWidget()

# trace 추가
fig.add_bar(
    x=pi.index,
    y=pi["무더위쉼터_합계"],
    name="무더위쉼터",
    marker_color="#ff7f0e"
)

fig.add_bar(
    x=pi.index,
    y=pi["지진대피소_합계"],
    name="지진대피소",
    marker_color="#1f77b4"
)

fig.add_bar(
    x=pi.index,
    y=pi["한파쉼터"],
    name="한파쉼터",
    marker_color="#17becf"
)

# 합계 텍스트용 trace
total_trace = go.Scatter(
    x=pi.index,
    y=pi["대피소_합계"],
    mode="text",
    text=pi["대피소_합계"].astype(str),
    textposition="top center",
    showlegend=False,
    textfont=dict(size=14, color="black")
)

fig.add_trace(total_trace)

fig.update_layout(
    barmode="stack",
    title="지역별 대피소 유형 및 총 대피소 수",
    xaxis_title="지역",
    yaxis_title="대피소 수",
    legend_title="대피소 유형",
    template="plotly_white"
)

def recompute_totals():
    totals = np.zeros(len(pi))

    for tr in fig.data[:3]:   # 앞 3개 bar trace만 계산
        # visible=True 또는 None 이면 보이는 상태
        if tr.visible is None or tr.visible is True:
            totals += np.array(tr.y, dtype=float)

    with fig.batch_update():
        fig.data[3].y = totals
        fig.data[3].text = [str(int(v)) if v > 0 else "" for v in totals]

def handle_trace_change(trace, points, state):
    recompute_totals()

# 각 bar trace에 콜백 연결
for tr in fig.data[:3]:
    tr.on_click(handle_trace_change)

fig

FigureWidget({
    'data': [{'marker': {'color': '#ff7f0e'},
              'name': '무더위쉼터',
              'type': 'bar',
              'uid': '96aa2e89-e45d-4a4c-bb58-9bf95d05426a',
              'x': array(['경남', '경북', '대구', '부산', '울산'], dtype=object),
              'y': {'bdata': 'Th4JFf4EWQZJBA==', 'dtype': 'i2'}},
             {'marker': {'color': '#1f77b4'},
              'name': '지진대피소',
              'type': 'bar',
              'uid': 'b30a9802-0b19-4b8a-965e-3420f4924cf0',
              'x': array(['경남', '경북', '대구', '부산', '울산'], dtype=object),
              'y': {'bdata': 'xgKIBtUCuAI3AQ==', 'dtype': 'i2'}},
             {'marker': {'color': '#17becf'},
              'name': '한파쉼터',
              'type': 'bar',
              'uid': 'fa955065-4502-4052-9f40-4adb423d07ff',
              'x': array(['경남', '경북', '대구', '부산', '울산'], dtype=object),
              'y': {'bdata': 'vwCLAFAALwCdAA==', 'dtype': 'i2'}},
             {'mode': 'text',
              'showlegend': False,
      

In [45]:
fig

FigureWidget({
    'data': [{'marker': {'color': '#ff7f0e'},
              'name': '무더위쉼터',
              'type': 'bar',
              'uid': '83ecd15e-e318-4b03-b1cc-1fa7c0f530f0',
              'x': array(['경남', '경북', '대구', '부산', '울산'], dtype=object),
              'y': {'bdata': 'Th4JFf4EWQZJBA==', 'dtype': 'i2'}},
             {'marker': {'color': '#1f77b4'},
              'name': '지진대피소',
              'type': 'bar',
              'uid': '58c45b2c-2843-44de-8150-5aece8c58b3b',
              'x': array(['경남', '경북', '대구', '부산', '울산'], dtype=object),
              'y': {'bdata': 'xgKIBtUCuAI3AQ==', 'dtype': 'i2'}},
             {'marker': {'color': '#17becf'},
              'name': '한파쉼터',
              'type': 'bar',
              'uid': 'cff44452-89d2-4326-bd84-238b3d9beb83',
              'x': array(['경남', '경북', '대구', '부산', '울산'], dtype=object),
              'y': {'bdata': 'vwCLAFAALwCdAA==', 'dtype': 'i2'}},
             {'mode': 'text',
              'showlegend': False,
      

### 지역별 재난 발생 횟수

In [ ]:
# 지역별 재난 발생 횟수
danger_count = (
    df.groupby("지역")
      .size()
      .reset_index(name="재난발생횟수")
)

# pi index를 컬럼으로 변환
pi_reset = pi.reset_index()

# 두 데이터 합치기
region_summary = danger_count.merge(
    pi_reset[["지역", "대피소_합계"]],
    on="지역",
    how="left"
)

region_summary

,지역,재난발생횟수,대피소_합계
0,경남,1486,8659
1,경북,1606,7196
2,대구,114,2083
3,부산,178,2368
4,울산,161,1565


: 

: 

In [ ]:
region_summary.columns

Index(['시도', '대피소수', '지역', '재난발생횟수'], dtype='object')

: 

: 

#### 지역별 재난 발생 횟수와 대피소 수 비교

In [ ]:
import plotly.express as px

fig = px.scatter(
    region_summary,
    x="재난발생횟수",
    y="대피소수",
    text="지역",
    size="재난발생횟수",
    color="지역",
    title="지역별 재난 발생 횟수와 대피소 수 비교",
    
    color_discrete_map={
        "경남": "#1f77b4",   # 파랑
        "경북": "#ff7f0e",   # 주황
        "대구": "#2ca02c",   # 초록
        "부산": "#d62728",   # 빨강
        "울산": "#9467bd"    # 보라
    }
)

fig.update_traces(
    textposition="top center",
    marker=dict(line=dict(width=1, color="black"))
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="재난 발생 횟수",
    yaxis_title="대피소 수"
)

fig.show()

: 

: 

In [ ]:
danger_type_region = pd.pivot_table(
    df,
    index="지역",
    columns="재난종류",
    aggfunc="size",
    fill_value=0
).reset_index()

danger_type_region

재난종류,지역,강풍,건조,대설,태풍,폭염,폭풍해일,한파,호우
0,경남,270,236,77,0,235,37,94,537
1,경북,299,224,152,19,265,0,187,460
2,대구,9,21,6,1,27,0,15,35
3,부산,66,22,0,0,24,12,8,46
4,울산,80,22,3,0,18,0,6,32


: 

: 

#### 지역별 재난종류 발생 패턴

In [ ]:
import plotly.express as px

heat = danger_type_region.set_index("지역")

text_values = heat.applymap(lambda x: "" if x == 0 else str(int(x)))

fig = px.imshow(
    heat,
    color_continuous_scale="YlOrRd",
    aspect="auto",
    title="지역별 재난종류 발생 패턴"
)

fig.update_traces(
    text=text_values,
    texttemplate="%{text}"
)

fig.update_layout(
    xaxis_title="재난종류",
    yaxis_title="지역"
)

fig.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_17564\2862703798.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  text_values = heat.applymap(lambda x: "" if x == 0 else str(int(x)))


: 

: 

In [ ]:
# 시도별 대피소 수

region_count = (
    df_sh.groupby("시도")
      .size()
      .reset_index(name="대피소수")
      .sort_values("대피소수", ascending=False)
)

region_count

,시도,대피소수
0,경남,8659
1,경북,7196
3,부산,2368
2,대구,2083
4,울산,1565


: 

: 

In [ ]:
danger_type_region

재난종류,지역,강풍,건조,대설,태풍,폭염,폭풍해일,한파,호우
0,경남,270,236,77,0,235,37,94,537
1,경북,299,224,152,19,265,0,187,460
2,대구,9,21,6,1,27,0,15,35
3,부산,66,22,0,0,24,12,8,46
4,울산,80,22,3,0,18,0,6,32


: 

: 

In [ ]:
# 시도별 대피소 수와 재난발생횟수 및 종류

danger_type_region = pd.pivot_table(
    df,
    index="지역",
    columns="재난종류",
    aggfunc="size",
    fill_value=0
).reset_index()

danger_count = (
    df.groupby("지역")
      .size()
      .reset_index(name="재난발생횟수")
)

region_count = (
    df_sh.groupby("시도")
      .size()
      .reset_index(name="대피소수")
      .sort_values("대피소수", ascending=False)
)

merged_df = (
    region_count
    .merge(danger_count, left_on="시도", right_on="지역", how="left")
    .merge(danger_type_region, left_on="시도", right_on="지역", how="left", suffixes=("", "_재난유형"))
    .drop(columns=["지역", "지역_재난유형"])
)

merged_df

,시도,대피소수,재난발생횟수,강풍,건조,대설,태풍,폭염,폭풍해일,한파,호우
0,경남,8659,1486,270,236,77,0,235,37,94,537
1,경북,7196,1606,299,224,152,19,265,0,187,460
2,부산,2368,178,66,22,0,0,24,12,8,46
3,대구,2083,114,9,21,6,1,27,0,15,35
4,울산,1565,161,80,22,3,0,18,0,6,32


: 

: 

#### 지역별 재난 종류 분포

In [ ]:
long_df = merged_df.melt(
    id_vars=["시도"],
    value_vars=["강풍","건조","대설","태풍","폭염","폭풍해일","한파","호우"],
    var_name="재난종류",
    value_name="발생건수"
)

fig = px.bar(
    long_df,
    x="시도",
    y="발생건수",
    color="재난종류",
    title="지역별 재난 종류 분포",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.update_layout(
    barmode="stack",
    template="plotly_white"
)

fig.show()

: 

: 

#### 지역별 재난 종류 발생 패턴

In [ ]:
heat = merged_df.set_index("시도")[[
    "강풍","건조","대설","태풍","폭염","폭풍해일","한파","호우"
]]

text_values = heat.applymap(lambda x: "" if x == 0 else str(int(x)))

fig = px.imshow(
    heat,
    color_continuous_scale="YlOrRd",
    aspect="auto",
    title="지역별 재난 종류 발생 패턴"
)

fig.update_traces(
    text=text_values,
    texttemplate="%{text}"
)

fig.update_layout(
    xaxis_title="재난종류",
    yaxis_title="지역"
)

fig.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_17564\2416677174.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  text_values = heat.applymap(lambda x: "" if x == 0 else str(int(x)))


: 

: 

In [ ]:
merged_df["대피소대비지수"] = (
    merged_df["대피소수"] / merged_df["재난발생횟수"]
).round(2)

merged_df

,시도,대피소수,재난발생횟수,강풍,건조,대설,태풍,폭염,폭풍해일,한파,호우,대피소대비지수
0,경남,8659,1486,270,236,77,0,235,37,94,537,5.83
1,경북,7196,1606,299,224,152,19,265,0,187,460,4.48
2,부산,2368,178,66,22,0,0,24,12,8,46,13.30
3,대구,2083,114,9,21,6,1,27,0,15,35,18.27
4,울산,1565,161,80,22,3,0,18,0,6,32,9.72


: 

: 

### 재난 대비 대피소 부족 지역 분석

In [ ]:
import plotly.express as px

risk_df = merged_df.sort_values("대피소대비지수", ascending=True)

fig = px.bar(
    risk_df,
    x="시도",
    y="대피소대비지수",
    color="대피소대비지수",
    text="대피소대비지수",
    color_continuous_scale="OrRd_r",
    title="재난 대비 대피소 부족 지역 분석"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    xaxis_title="지역",
    yaxis_title="대피소수 / 재난발생횟수"
)

fig.show()

: 

: 

In [ ]:
avg_ratio = merged_df["대피소대비지수"].mean()

merged_df["위험등급"] = merged_df["대피소대비지수"].apply(
    lambda x: "부족" if x < avg_ratio else "양호"
)

merged_df

,시도,대피소수,재난발생횟수,강풍,건조,대설,태풍,폭염,폭풍해일,한파,호우,대피소대비지수,위험등급
0,경남,8659,1486,270,236,77,0,235,37,94,537,5.83,부족
1,경북,7196,1606,299,224,152,19,265,0,187,460,4.48,부족
2,부산,2368,178,66,22,0,0,24,12,8,46,13.30,양호
3,대구,2083,114,9,21,6,1,27,0,15,35,18.27,양호
4,울산,1565,161,80,22,3,0,18,0,6,32,9.72,부족


: 

: 

#### 재난 대비 대피소 부족 지역 탐지

In [ ]:
import plotly.express as px

merged_df["대피소대비지수"] = (
    merged_df["대피소수"] / merged_df["재난발생횟수"]
).round(2)

avg_ratio = merged_df["대피소대비지수"].mean()

merged_df["위험등급"] = merged_df["대피소대비지수"].apply(
    lambda x: "부족" if x < avg_ratio else "양호"
)

fig = px.bar(
    merged_df.sort_values("대피소대비지수"),
    x="시도",
    y="대피소대비지수",
    color="위험등급",
    text="대피소대비지수",
    color_discrete_map={
        "부족": "#d62728",
        "양호": "#1f77b4"
    },
    title="재난 대비 대피소 부족 지역 탐지"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    xaxis_title="지역",
    yaxis_title="대피소수 / 재난발생횟수"
)

fig.show()

: 

: 

: 

: 